# Kaggle Playground Series: Exploring Mental Health Data

### Project Prerequisites

In [9]:
import warnings
import numpy as np
import pandas as pd
import xgboost as xgb
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import RandomizedSearchCV,GridSearchCV

In [10]:
warnings.filterwarnings("ignore")

### Preparing the Data

In [50]:
df = pd.read_csv("Data/train.csv")
df.set_index("id",inplace=True)
df.drop(columns = ["Name","City"],inplace=True)
df.head()

,Gender,Age,Working Professional or Student,Profession,Academic Pressure,Work Pressure,CGPA,Study Satisfaction,Job Satisfaction,Sleep Duration,Dietary Habits,Degree,Have you ever had suicidal thoughts ?,Work/Study Hours,Financial Stress,Family History of Mental Illness,Depression
id,,,,,,,,,,,,,,,,,
0,Female,49.0,Working Professional,Chef,NaN,5.0,NaN,NaN,2.0,More than 8 hours,Healthy,BHM,No,1.0,2.0,No,0
1,Male,26.0,Working Professional,Teacher,NaN,4.0,NaN,NaN,3.0,Less than 5 hours,Unhealthy,LLB,Yes,7.0,3.0,No,1
2,Male,33.0,Student,NaN,5.0,NaN,8.97,2.0,NaN,5-6 hours,Healthy,B.Pharm,Yes,3.0,1.0,No,1
3,Male,22.0,Working Professional,Teacher,NaN,5.0,NaN,NaN,1.0,Less than 5 hours,Moderate,BBA,Yes,10.0,1.0,Yes,1
4,Female,30.0,Working Professional,Business Analyst,NaN,1.0,NaN,NaN,1.0,5-6 hours,Unhealthy,BBA,Yes,9.0,4.0,Yes,0


In [51]:
df.isnull().sum()

Gender                                        0
Age                                           0
Working Professional or Student               0
Profession                                36630
Academic Pressure                        112803
Work Pressure                             27918
CGPA                                     112802
Study Satisfaction                       112803
Job Satisfaction                          27910
Sleep Duration                                0
Dietary Habits                                4
Degree                                        2
Have you ever had suicidal thoughts ?         0
Work/Study Hours                              0
Financial Stress                              4
Family History of Mental Illness              0
Depression                                    0
dtype: int64

In [52]:
def Feature_Handling(x):
    x["Gender"] = x["Gender"].apply(lambda x: 1 if x=="Female" else 0)
    x["Have you ever had suicidal thoughts ?"] = x["Have you ever had suicidal thoughts ?"].apply(lambda x: 1 if x=="Yes" else 0)
    x["Family History of Mental Illness"] = x["Family History of Mental Illness"].apply(lambda x: 1 if x=="Yes" else 0)
    x["Working Professional or Student"] = x["Working Professional or Student"].apply(lambda x:1 if x=="Student" else 0)
    x = pd.concat([x,pd.get_dummies(x["Profession"],prefix="Profession").astype("int")],axis=1)
    x = pd.concat([x,pd.get_dummies(x["Dietary Habits"],prefix="Dietary_Habits").astype("int")],axis=1)
    x = pd.concat([x,pd.get_dummies(x["Degree"],prefix="Degree").astype("int")],axis=1)
    x = pd.concat([x,pd.get_dummies(x["Sleep Duration"],prefix="Sleep_Duration").astype("int")],axis=1)
    x.drop(columns=["Profession","Dietary Habits","Degree","Sleep Duration"],inplace = True)
    return x

df = Feature_Handling(df)
df.head()

,Gender,Age,Working Professional or Student,Academic Pressure,Work Pressure,CGPA,Study Satisfaction,Job Satisfaction,Have you ever had suicidal thoughts ?,Work/Study Hours,...,Sleep_Duration_Indore,Sleep_Duration_Less than 5 hours,Sleep_Duration_Moderate,Sleep_Duration_More than 8 hours,Sleep_Duration_No,Sleep_Duration_Pune,Sleep_Duration_Sleep_Duration,Sleep_Duration_Unhealthy,Sleep_Duration_Work_Study_Hours,Sleep_Duration_than 5 hours
id,,,,,,,,,,,,,,,,,,,,,
0,1,49.0,0,NaN,5.0,NaN,NaN,2.0,0,1.0,...,0,0,0,1,0,0,0,0,0,0
1,0,26.0,0,NaN,4.0,NaN,NaN,3.0,1,7.0,...,0,1,0,0,0,0,0,0,0,0
2,0,33.0,1,5.0,NaN,8.97,2.0,NaN,1,3.0,...,0,0,0,0,0,0,0,0,0,0
3,0,22.0,0,NaN,5.0,NaN,NaN,1.0,1,10.0,...,0,1,0,0,0,0,0,0,0,0
4,1,30.0,0,NaN,1.0,NaN,NaN,1.0,1,9.0,...,0,0,0,0,0,0,0,0,0,0


In [53]:
def HandlingNan(df):
    df["Academic Pressure"] = df["Academic Pressure"].fillna(df["Academic Pressure"].median())
    df["Study Satisfaction"] = df["Study Satisfaction"].fillna(df["Study Satisfaction"].median())
    df["Job Satisfaction"] = df["Job Satisfaction"].fillna(df["Job Satisfaction"].median())
    df["Work Pressure"] = df["Work Pressure"].fillna(df["Work Pressure"].median())
    df["CGPA"] = df["CGPA"].fillna(df["CGPA"].mean())
    return df

df = HandlingNan(df)
df.head()

,Gender,Age,Working Professional or Student,Academic Pressure,Work Pressure,CGPA,Study Satisfaction,Job Satisfaction,Have you ever had suicidal thoughts ?,Work/Study Hours,...,Sleep_Duration_Indore,Sleep_Duration_Less than 5 hours,Sleep_Duration_Moderate,Sleep_Duration_More than 8 hours,Sleep_Duration_No,Sleep_Duration_Pune,Sleep_Duration_Sleep_Duration,Sleep_Duration_Unhealthy,Sleep_Duration_Work_Study_Hours,Sleep_Duration_than 5 hours
id,,,,,,,,,,,,,,,,,,,,,
0,1,49.0,0,3.0,5.0,7.658636,3.0,2.0,0,1.0,...,0,0,0,1,0,0,0,0,0,0
1,0,26.0,0,3.0,4.0,7.658636,3.0,3.0,1,7.0,...,0,1,0,0,0,0,0,0,0,0
2,0,33.0,1,5.0,3.0,8.970000,2.0,3.0,1,3.0,...,0,0,0,0,0,0,0,0,0,0
3,0,22.0,0,3.0,5.0,7.658636,3.0,1.0,1,10.0,...,0,1,0,0,0,0,0,0,0,0
4,1,30.0,0,3.0,1.0,7.658636,3.0,1.0,1,9.0,...,0,0,0,0,0,0,0,0,0,0
